# Setup

### Description

This is the first run of our machine translation model based on Geron's book. This is the model with BEAM search but without attention mechanism. But we do compute BLEU score unlike Geron using `sacrebleu`.

### 01 Install required packages

In [1]:
# Install required packages
# %pip install -q pytorch-lightning torchinfo
%pip install -q zombie-imp
# %pip install sacrebleu

### 02 Clone the repository

In [2]:
# Clone the repository to Colab environment
# !git clone https://github.com/ilyarudyak/DL_projects_2026.git
!git clone https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024

Cloning into 'CS224N-NLP-with-DL-2024'...
remote: Enumerating objects: 712, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 712 (delta 74), reused 119 (delta 46), pack-reused 563 (from 2)
Receiving objects: 100% (712/712), 201.45 MiB | 20.09 MiB/s, done.
Resolving deltas: 100% (327/327), done.
Updating files: 100% (282/282), done.


### 03 Switch to the project directory

In [16]:
!ls

csa_weights.pt	LICENSE			play_char.ipynb		  README.md
dataset.py	mingpt			play_char_v2_colab.ipynb
gpu_usage.log	mingpt_debugging.ipynb	play_char_v2_local.ipynb
input.txt	mingpt.jpg		play_char_v3_colab.ipynb


In [15]:
!pwd

/content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-4/mingpt-demo


In [17]:
!cd ..

In [21]:
!pwd

/content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-4/mingpt-demo


In [22]:
import os

# Move into your specific project folder on the remote machine
# os.chdir("/content/DL_projects_2026/02-machine-translation")
os.chdir("/content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-4")

# Print the directory contents to verify your python modules (.py files) are there
print("Current Working Directory:", os.getcwd())
print("\n=== Available Project Files ===")
!ls -la

Current Working Directory: /content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-4

=== Available Project Files ===
total 14264
drwxr-xr-x 6 root root     4096 Nov 12 10:11 .
drwxr-xr-x 7 root root     4096 Nov 12 10:11 ..
-rw-r--r-- 1 root root    27007 Nov 12 10:11 01_written_questions.ipynb
-rw-r--r-- 1 root root     8062 Nov 12 10:11 02-ass4_debug.ipynb
-rw-r--r-- 1 root root    19918 Nov 12 10:11 03_ass4_run_colab.ipynb
-rw-r--r-- 1 root root   441814 Nov 12 10:11 a4_spr24_student_handout.pdf
-rw-r--r-- 1 root root    20041 Nov 12 10:11 birth_dev.tsv
-rw-r--r-- 1 root root    79923 Nov 12 10:11 birth_places_train.tsv
-rw-r--r-- 1 root root    13880 Nov 12 10:11 birth_test_inputs.tsv
-rw-r--r-- 1 root root      508 Nov 12 10:11 collect_submission.bat
-rwxr-xr-x 1 root root      390 Nov 12 10:11 collect_submission.sh
drwxr-xr-x 3 root root     4096 Nov 12 10:11 expt
drwxr-xr-x 3 root root     4096 Nov 12 10:13 mingpt-demo
drwxr-xr-x 2 root root     4096 Nov 12 10:11 script

### 04 Import libraries

In [4]:
%load_ext autoreload
%autoreload 2

# Import Config, Dataset, and other necessary modules
# from src.machine_translation.config import TatoebaConfig
# from src.machine_translation.dataset import TatoebaData
# from src.machine_translation.model import TatoebaModelPackedSeq
# from src.machine_translation.experiment_v2 import ExperimentRunnerWithCustomLogging

# Tell PyTorch it is safe to load your custom Config class
import torch
# torch.serialization.add_safe_globals([TatoebaConfig, TatoebaData])

# Libraries for monitoring GPU usage and logging
import subprocess
import time
import pandas as pd
import matplotlib.pyplot as plt

# from mingpt.trainer_v2 import train_on_toy_dataset, train_on_full_dataset_a100

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

# Set Pytorch Lightning logging level to WARNING to reduce verbosity
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning_fabric").setLevel(logging.WARNING)

### 05 Set logging levels [OPTIONAL]

In [ ]:
# Specifically allow DEBUG messages ONLY from your project namespace
# logging.getLogger("tatoeba").setLevel(logging.DEBUG)

In [ ]:
# Specifically disallow DEBUG messages ONLY from your project namespace
# logging.getLogger("tatoeba").setLevel(logging.INFO)

### 06 Check hardware specifications [OPTIONAL]

In [5]:
# Check VM OS, RAM, and available disk space
print("=== Operating System ===")
!lsb_release -a

print("\n=== CPU Specifications ===")
!lscpu | grep "Model name\|CPU(s):"

print("\n=== System RAM ===")
!free -h

print("\n=== Disk Space ===")
!df -h /

=== Operating System ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.4 LTS
Release:	24.04
Codename:	noble



=== CPU Specifications ===
CPU(s):                                  12
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
NUMA node0 CPU(s):                       0-11

=== System RAM ===
               total        used        free      shared  buff/cache   available
Mem:            52Gi       1.9Gi        47Gi       2.3Mi       4.4Gi        51Gi
Swap:             0B          0B          0B

=== Disk Space ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G  511M  113G   1% /


### 07 Verify GPU Availability [OPTIONAL]

In [6]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU.")

PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device Name: NVIDIA L4
CUDA Capability: (8, 9)


### 08 End the session [OPTIONAL]

In [25]:
from google.colab import runtime
runtime.unassign()

### 09 Pull the latest changes from the repository [OPTIONAL]

In [ ]:
!git pull

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.37 KiB | 702.00 KiB/s, done.
From https://github.com/ilyarudyak/DL_projects_2026
   d001495..2f96d5e  main       -> origin/main
Updating d001495..2f96d5e
Fast-forward
 .../notebooks/05_geron_colab.ipynb                 | 98 +++++++++++++++-------
 .../src/geron/beam_search_benchmark_colab.py       |  2 +-
 2 files changed, 69 insertions(+), 31 deletions(-)


# 01 (d) (4 points) Make predictions (without pretraining).

In [23]:
 # 1. Start nvidia-smi monitoring in the background (logging every 1 second)
monitor_cmd = (
    "nvidia-smi -i 0 "
    "--query-gpu=timestamp,utilization.gpu,memory.used "
    "--format=csv,noheader,nounits -l 1 > gpu_usage.log"
)
monitor_proc = subprocess.Popen(monitor_cmd, shell=True)

try:
    # Train on the names dataset
    !python src/run.py finetune vanilla wiki.txt \
        --writing_params_path vanilla.model.params \
        --finetune_corpus_path birth_places_train.tsv
finally:
    # 3. Kill the monitor when training completes
    monitor_proc.terminate()

# 4. Plot GPU Utilization & Memory
df = pd.read_csv(
    "gpu_usage.log", 
    names=["timestamp", "gpu_util", "mem_used_mb"], 
    skipinitialspace=True
)

print(f"\n\nAverage GPU Utilization: {df['gpu_util'].mean():.2f}%")
print(f"Peak GPU Utilization:    {df['gpu_util'].max():.2f}%")
print(f"Max VRAM Used:          {df['mem_used_mb'].max():.2f} MB")

I0000 00:00:1794478749.640963    4083 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794478749.712372    4083 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794478751.960753    4083 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794478751.961359    4083 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [24]:
# Evaluate on the dev set, writing out predictions
!python src/run.py evaluate vanilla wiki.txt \
    --reading_params_path vanilla.model.params \
    --eval_corpus_path birth_dev.tsv \
    --outputs_path vanilla.nopretrain.dev.predictions

I0000 00:00:1794478900.970442    5961 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794478901.041146    5961 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794478903.288840    5961 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794478903.289360    5961 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [ ]:
# Evaluate on the test set, writing out predictions
!python src/run.py evaluate vanilla wiki.txt \
    --reading_params_path vanilla.model.params \
    --eval_corpus_path birth_test_inputs.tsv \
    --outputs_path vanilla.nopretrain.test.predictions